# 10 — Stage A: public multitask training — Apple Silicon

Trains jev-lite on the Stage A mix (`train.e0`, `train.e1`), evaluates on `test_in` / `test_ood` and saves the final model. All logic is in `jevlite.train` (launched by `scripts/train.py`); this notebook only differs from its twin in the parameters cell.

- **Resume:** checkpoints every `train.checkpoint.every_minutes` (40), mirrored to the private HF repo `train.checkpoint.hub_repo`. If a session ends, run the notebook again with the same `RUN`: it continues from the newest checkpoint.
- **Kaggle settings:** Accelerator GPU T4 x2, Internet On, input dataset `jevlite-mix`, secret `HF_TOKEN` attached (Add-ons → Secrets).
- **Smoke run first:** `MAX_STEPS = 200` checks speed, memory, the eval and a Hub upload in a few minutes. Then set it to `None` and use Save & Run All for the real run.
- **Both versions:** run once with `MIX = "stage_a"` and once with `"stage_a_nosa"` (and a different `RUN`).

## 0. Install (Kaggle / Colab)

In [ ]:
# Kaggle / Colab: install the code from GitHub at a fixed commit. A no-op on a local checkout.
import os, sys

COMMIT = "main"   # set a commit hash for a reproducible run
ON_CLOUD = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or "COLAB_RELEASE_TAG" in os.environ
if ON_CLOUD:
    !rm -rf /tmp/JEV_Judge && git clone -q https://github.com/Nefelim584/JEV_Judge.git /tmp/JEV_Judge
    !cd /tmp/JEV_Judge && git checkout -q {COMMIT} && git log -1 --format='%h %ad %s' --date=short
    !pip install -q --no-deps --ignore-requires-python -e /tmp/JEV_Judge
    !pip install -q loguru peft bitsandbytes
    !pip uninstall -y -q torchao  # the image has torchao 0.10; recent peft refuses < 0.16 instead of skipping it
    sys.path.insert(0, "/tmp/JEV_Judge/src")  # the running kernel does not pick up a fresh editable install
!python --version

## 1. Parameters

The only stack-specific cell.

In [ ]:
STACK = "apple_silicon"
RUN = "stage_a-lora-s42"     # one folder locally and on the Hub; the same name resumes the run
MIX = "stage_a"              # "stage_a_nosa" for the no-SA version (same hyperparameters)
DATA_ROOT = "../../data/mix"
RUNS_DIR = "../../runs"
OUT = "../../runs"
MAX_MINUTES = None
MAX_STEPS = None             # e.g. 200 for a smoke run (ends with an eval and a checkpoint)
HUB = True                   # mirror checkpoints to train.checkpoint.hub_repo (needs HF_TOKEN)

## 2. Environment: HF token, devices, paths

In [ ]:
import os
from pathlib import Path

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ and HUB:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF token:", "set" if os.environ.get("HF_TOKEN") else "missing (hub uploads will fail)" if HUB else "not needed")

import torch
import jevlite

REPO = Path(jevlite.__file__).resolve().parents[2]
NPROC = max(torch.cuda.device_count(), 1)
print("repo:", REPO, "| processes:", NPROC, "| data:", sorted(p.name for p in (Path(DATA_ROOT) / MIX).glob("*.jsonl")))

## 3. Train

One process per GPU (`torchrun`). Logs stream here and go to `RUNS_DIR/RUN/run.log`.

In [ ]:
args = [
    f"--stack {STACK}", f"--run {RUN}", f"--data-root {DATA_ROOT}", f"--runs-dir {RUNS_DIR}",
    f"--set train.stage_a.mix=mixes/{MIX}.yaml",
]
if MAX_MINUTES:
    args.append(f"--max-minutes {MAX_MINUTES}")
if MAX_STEPS:
    args.append(f"--max-steps {MAX_STEPS}")
if not HUB:
    args.append("--no-hub")
cmd = f"torchrun --standalone --nproc_per_node {NPROC} {REPO}/scripts/train.py " + " ".join(args)
print(cmd)
!{cmd}

## 4. Reports

Training curves always; the full eval report only once the run is done (`preds.jsonl` exists). Copied to `OUT` for download.

In [ ]:
import shutil

run_dir = Path(RUNS_DIR) / RUN
out = Path(OUT) / RUN
out.mkdir(parents=True, exist_ok=True)
for name in ("metrics.jsonl", "run.log", "config.yaml"):
    if (run_dir / name).exists():
        shutil.copy(run_dir / name, out / name)
!python {REPO}/scripts/make_report.py training --run {RUN}={run_dir} --out {out}/training
if (run_dir / "preds.jsonl").exists():
    data = f"{DATA_ROOT}/{MIX}/test_in.jsonl {DATA_ROOT}/{MIX}/test_ood.jsonl"
    !python {REPO}/scripts/eval.py --data {data} --pred {run_dir}/preds.jsonl --out {out}/eval --bootstrap 1000
    print((out / "eval" / "report.md").read_text()[:3000])
else:
    print("run not finished: re-run the notebook with the same RUN to resume")